## Libraries

In [11]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors

## Config

In [12]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_START_DATE = pd.Timestamp("2007-04-01")
TRAIN_END_DATE   = pd.Timestamp("2022-03-31")
TEST_START_DATE  = pd.Timestamp("2022-04-01")   # all dates >= this are test

# <<< FILL THESE FROM CV RESULTS >>>
best_lag_set = [1, 12]   
best_params  = {
    "max_iter": 600,
    "max_depth": 3,
    "learning_rate": 0.1,
    "max_leaf_nodes": 31,
    "min_samples_leaf": 50,
    "l2_regularization": 1.0,
}

# continuous features
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4"
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

## Metric functions

In [13]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

# def morans_i(residuals, xs, ys, k=5, eps=1e-8):
#     """
#     Simple Moran's I using k-nearest neighbours with inverse-distance weights.
#     residuals: shape (N,)
#     xs, ys: coordinates aligned with residuals.
#     """
#     residuals = np.asarray(residuals)
#     N = len(residuals)
#     x_mean = residuals.mean()
#     x_dev = residuals - x_mean

#     coords = np.column_stack([xs, ys])
#     nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
#     distances, indices = nbrs.kneighbors(coords)

#     W = np.zeros((N, N))
#     for i in range(N):
#         neigh_idx = indices[i, 1:]  # skip self
#         w = 1.0 / (distances[i, 1:] + eps)
#         W[i, neigh_idx] = w

#     S0 = W.sum()
#     num = 0.0
#     for i in range(N):
#         for j in range(N):
#             num += W[i, j] * x_dev[i] * x_dev[j]
#     den = np.sum(x_dev ** 2) + eps
#     I = (N / S0) * (num / den)
#     return I

In [14]:
def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.

    Parameters
    ----------
    residuals : array-like, shape (N,)
        Values (e.g. residuals) for each spatial unit.
    xs, ys : array-like, shape (N,)
        Coordinates (e.g. centroids) aligned with residuals.
    k : int, default=5
        Number of nearest neighbours to use.
    eps : float, default=1e-8
        Small constant to avoid division by zero.
    symmetric : bool, default=True
        If True, symmetrise the weight matrix: W = 0.5 * (W + W.T).
    row_standardize : bool, default=True
        If True, row-standardise W so each row sums to 1.
        This makes Moran's I more comparable across runs.
    permutations : int, default=0
        Number of random permutations to use for significance testing.
        If 0, no permutation test is performed.
    random_state : int or np.random.Generator, optional
        Seed or Generator for permutations.

    Returns
    -------
    result : dict
        {
          "I": observed Moran's I,
          "S0": sum of weights,
          "permutations": array of permuted I values (if permutations > 0),
          "z_score": z-score of observed I (if permutations > 0),
          "p_value": two-sided p-value (if permutations > 0)
        }
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        # Avoid division by zero for isolated nodes
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    # num = sum_ij w_ij * (x_i - x̄)(x_j - x̄)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        # Note: den is invariant under permutation (same deviations squared)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        # two-sided p-value
        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result


## Load data

In [15]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"')

df = df.sort_values([ENTITY_COL, TIME_COL])


## Rolling STL feature builder

In [16]:
def add_rolling_stl_components(
    df,
    entity_col,
    time_col,
    target_col,
    period=12,
    min_history=24,
    window=120,        # set None for expanding, or e.g. 120 to match your 10y window
    robust=True,
    show_progress=True,
):
    """
    For each LA, compute STL components at time t using only y up to time t.
    We assign the *last* STL values from the fitted history to that time t.

    IMPORTANT:
    - This creates stl_trend/stl_seasonal/stl_resid for each row.
    - You should only use *lags* of these components (e.g., lag1/lag12/lag24)
      when predicting y_t, otherwise you'd leak y_t into its own features.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    # Precompute total iterations for tqdm
    groups = list(df.groupby(entity_col))
    total_steps = sum(len(sub) for _, sub in groups)

    iterator = tqdm(
        groups,
        desc="Rolling STL per LA",
        total=len(groups),
        leave=True,
        disable=not show_progress,
    )

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        n = len(sub)

        for t in range(n):
            start = 0 if window is None else max(0, t - window + 1)
            hist = y[start : t + 1]

            if len(hist) < min_history or np.isnan(hist).any():
                continue

            try:
                res = STL(hist, period=period, robust=robust).fit()

                idx = sub.index[t]
                df.loc[idx, "stl_trend"]    = res.trend[-1]
                df.loc[idx, "stl_seasonal"] = res.seasonal[-1]
                df.loc[idx, "stl_resid"]    = res.resid[-1]

            except Exception:
                continue

    return df

## Evaluation

In [17]:
df = add_rolling_stl_components(
    df,
    entity_col=ENTITY_COL,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    period=12,
    min_history=24,
    window=120,   # <- recommend matching your CV training window
    robust=True,
)

# =========================================================
# LAGGED STL FEATURES (best_lag_set) ACROSS FULL PANEL
# =========================================================
required_lag_cols = []
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        col = f"{comp}_lag{lag}"
        df[col] = df.groupby(ENTITY_COL)[comp].shift(lag)
        required_lag_cols.append(col)

# =========================================================
# TRAIN / TEST SPLIT (feature period starts April 2007)
# =========================================================
mask_train = (df[TIME_COL] >= TRAIN_START_DATE) & (df[TIME_COL] <= TRAIN_END_DATE)
mask_test  = (df[TIME_COL] >= TEST_START_DATE)

df_train = df.loc[mask_train].copy()
df_test  = df.loc[mask_test].copy()

# Need all lag columns present
df_train = df_train.dropna(subset=required_lag_cols)
df_test  = df_test.dropna(subset=required_lag_cols)

# =========================================================
# STANDARDISE CONTINUOUS + LAG FEATURES
# =========================================================
scale_cols = continuous_cols + required_lag_cols
scaler = StandardScaler()
df_train[scale_cols] = scaler.fit_transform(df_train[scale_cols])
df_test[scale_cols]  = scaler.transform(df_test[scale_cols])

# =========================================================
# BUILD MATRICES
# =========================================================
X_train = df_train[continuous_cols + categorical_cols + required_lag_cols]
y_train = df_train[TARGET_COL].values

X_test  = df_test[continuous_cols + categorical_cols + required_lag_cols]
y_test  = df_test[TARGET_COL].values

# =========================================================
# TRAIN FINAL RANDOM FOREST
# =========================================================
gb = HistGradientBoostingRegressor(
        **best_params,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        tol=1e-4,
        random_state=42
    )
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

df_test["y_pred"] = y_pred

# =========================================================
# GLOBAL ACCURACY
# =========================================================
global_mae   = mae(y_test, y_pred)
global_rmse  = rmse(y_test, y_pred)
global_smape = smape(y_test, y_pred)
global_mase  = mase(y_test, y_pred, y_train, m=12)

print("=== Global accuracy ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# =========================================================
# ACROSS-LA CONSISTENCY
# =========================================================
la_mae = (
    df_test
    .groupby(ENTITY_COL)
    .apply(lambda g: mae(g[TARGET_COL], g["y_pred"]))
)

median_mae = la_mae.median()
p75_mae    = la_mae.quantile(0.75)

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# SPATIO-TEMPORAL DIAGNOSTICS
# =========================================================
# Moran's I: mean residual per LA over test period
df_test["resid"] = df_test[TARGET_COL] - df_test["y_pred"]

# Mean residual per LA over test period
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

# --- Robust centroid extraction (avoids pre-2007 NaNs) ---
centroids = (
    df_test
        .dropna(subset=["centroid_x", "centroid_y"])   # remove rows with missing centroids
        .sort_values(TIME_COL)
        .groupby(ENTITY_COL)
        .tail(1)                                       # take latest valid centroid per LA
        .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
        .loc[la_resid_mean.index]
)

# --- SAFETY FILTER (this is the mask you were missing) ---
mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

# Optional sanity check
print(f"LAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

# --- Compute Moran's I safely ---
N_valid = len(centroids_valid)
if N_valid <= 1:
    I_moran = np.nan
else:
    k_effective = min(5, N_valid - 1)

    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=5,
        eps=1e-8,
        symmetric=True,
        row_standardize=True,
        permutations=999,
        random_state=42,
    )

        
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran["I"]:.4f}")
    print(f"Moran's I z score: {I_moran["z_score"]:.4f}")
    print(f"Moran's I p value: {I_moran["p_value"]:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = (
    df_test
    .groupby(TIME_COL)["resid"]
    .mean()
    .sort_index()
)

lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)

# Extract the first (and only) row as floats
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])

print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# OPTIONAL: DIRECTIONAL ACCURACY & GROWTH-RATE ERROR
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


Rolling STL per LA: 100%|██████████| 294/294 [05:18<00:00,  1.08s/it]


=== Global accuracy ===
MAE   : 6,243.855
RMSE  : 12,151.434
sMAPE : 1.797%
MASE  : 0.297

=== Across-LA consistency ===
Median LA MAE       : 4,886.584
75th percentile MAE : 6,360.746
LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): -0.2439
Moran's I z score: -7.8844
Moran's I p value: 0.0010
Ljung–Box Q(12): stat=56.208, p=0.0000

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.572
Growth-rate error MAE (12-month)  : 0.0242


C:\Users\slong\AppData\Local\Temp\ipykernel_33008\105022710.py:88: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: mae(g[TARGET_COL], g["y_pred"]))


In [18]:
print("TEST rows:", len(df_test))
print("TEST date range:", df_test[TIME_COL].min(), "->", df_test[TIME_COL].max())
print("Unique months:", df_test[TIME_COL].nunique())
print("Unique LAs:", df_test[ENTITY_COL].nunique())

TEST rows: 7056
TEST date range: 2022-04-01 00:00:00 -> 2024-03-01 00:00:00
Unique months: 24
Unique LAs: 294


## Result output

In [19]:

output_path = "../../results/gb_final_test_results.xlsx"

summary_df = pd.DataFrame([{
    "model_type": "Gradient Boosting",
    "lag_set": str(best_lag_set),
    "params": str(best_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": I_moran,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")



Results saved to: ../../results/gb_final_test_results.xlsx


In [20]:
print("TEST rows:", len(df_test))
print("TEST date range:", df_test[TIME_COL].min(), "->", df_test[TIME_COL].max())
print("Unique months:", df_test[TIME_COL].nunique())
print("Unique LAs:", df_test[ENTITY_COL].nunique())

TEST rows: 7056
TEST date range: 2022-04-01 00:00:00 -> 2024-03-01 00:00:00
Unique months: 24
Unique LAs: 294
